# Lecture 7 · Let's build GPT

ChatGPT is a probabilistic system, and for any one prompt, it cna give us multiple answers sorts of replying to it.

GPT is short for `generatively pre-trained transformer`.

So **transformer** is the neural net that actually does all the heavy lifting under the hood.

What I'd like to focus on is just to train a transformer-based language model, and in our case, it's going to be a character-level language model.

We need a small data set, in this case, I propose that we work with `Tiny Shakespeare`. And it's all Shakespeare in a single file.

And what we are going to do now is we're going to basically model how these characters follow each other.

And this is just coming out of the transformer in a very similar manner as it would come out in ChatGPT, in our case, character by character, in ChatGPT, it's coming out on the token by token level, and token are these sort of like little subword pieces. So they are not word level, they're kind of like word chunk level.

In [3]:
# read it in to inspect it
with open('Lecture-7-input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [4]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  1115394


In [5]:
# let's look at the first 1000 characters
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



Next we are going to take this text and the text is a sequence of characters in Python. 

So I call the set constructor on it, I'm just going to get the set of all the characters that occur in this text. And then I call list on that to create a list of those characters instead of just a set so that I have a ordering, an arbitrary ordering. And then I sort that.

So basically we get just all the characters that occur in the entire dataset and they're sorted.

And the number of them is going to be our vocabulary size, these are the possible elements of our sequences. And the vocab is also the sort of possible characters that the model can see or emit.

In [6]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


So the next we would like to develop some strategy to tokenize the input text.

Now, when people say tokenize, they mean convert the raw text as a string to some sequence of integers according to some vocabulary of possible elements.

So as an example, here we are going to be building a character level language model. So we are going to be translating individual characters into integers.

In [7]:
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


So we are building both the encoder and decoder.

When we encode an arbitrary text, like hii there, we are going to receive a list of integers that represents that string. And we also have reverse mapping. And we can take this list and decode it to get back the exact same string. So it's really just like a translation to integers and back for arbitrary string. And for us, it is done a character level.

The way this is achieved is we just iterate over all the characters here and create a look up table from the character to integer and vice versa. And then to encode some string, we simply translate all the characters individually and to decode it back, we use the reverse mapping and concatenate all of it.

But there's many other schemas that people have come up with in practice.
So for example, Google uses a `sentence piece`, `sentence piece` will also encode text into integers, but in a different schema and using a different vocabulary. And `sentence piece` is a subword sort of tokenizer. And what that means is that you're not encoding entire words, but you're not also encoding individual characters. It's a subword unit level.

Also OpenAI has this library called `TikToken` that uses a byte pair encoding tokenizer, and that's what GPT uses.

So now we have an encoder and a decoder, effectively a tokenizer, we can tokenize the entire training set of Shakespeare.

So we are going to take all of text in tiny Shakespeare, encode it, and then wrap it into a torch.tensor to get the data tensor.

In [8]:
# let's now encode the entire text dataset and store it into a torch.tensor
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000]) # the 1000 characters we looked at earier will to the GPT look like this.

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

We see that we have a massive sequence of integers and this sequence of integers here is basically an identical translation of first 1000 characters here. And from now on the entire dataset of text is represented as, it's just stretched out as a single, a very large sequence of integers.

Let me do one more thing before we more on here.

I'd like to separate out our dataset into a train and a validation split.

So in particular, we're going to take the first 90% of the data set and consider that to be the training data for the transformer. And we're going to withhold the last 10% at the end of it to be the validation data. And this will help us understand to what extent our model is overfitting.

So we're going to basically hide and keep the validation data on the side. Because we don't want just a perfect memorization of this exact Shakespeare, we want a neural network that sort of creates Shakespeare-like text and so it should be fairly likely for it to produce the actual stowed away true Shakespeare text and so we're going to use this to get a sense of the overfitting.

In [9]:
# let's now split up the data into train and validation sets
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

So now we would like to start plugging these text sequences or integer sequences into the transformer so that it can train and learn those patterns.

Now the important thing to realize is we're never going to actually feed entire text into your transformer at all once. That would be computationally very expensive and prohibitive. So when we actually train a transformer on a lot of these datasets, we only work with chunks of the dataset. And when we train the transformer, we basically sample random little chunks out of the training set and train them just chunks at a time. And these chunks have basically some kind of a length and some maximum length.

Now the maximum length typically, at least in the code I usually write, is called block size. You can find it under different names like context length or something like that.

Let's start with the block size of just 8.

And let me look at the first train data characters, the first block size plus one characters.

In [10]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

So this is the first nine characters in the sequence, in the training set.

Now what I like to point out is that when you sample a chunk of data like this, so say these nine characters out of the train set, this actually has multiply examples packed into it. And that's because all of these characters follow each other. And so what this thing is going to say when we plug it into transformer is going to actually simultaneously train it to make prediction at every one of these positions.

Now in a chunk of nine characters, there's actually eight individual examples packed in there. So there's the example that in the context of 18, 47 likely comes next. In the context of 18 and 47, 56 comes next. In the context of 18, 47 and 56, 57 comes next, and so on. So that's the eight individual examples.

Let me spell it out with code.

**x** are the inputs of the transformer. It will just be the first block size characters. **y** will be the next block size of characters. So it offset by one. And that's because **y** are the targets for each position in the input. And then here I'm iterating over all the block size of eight. And the context is always all the characters in **x** up to **t** and including **t**. And the target is always the **t** character, but in the target's array **y**

In [13]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


These are the eight examples hidden in a chunk of nine characters that we sampled from the training set.

I also mention one more thing.

We train on all the eight examples here with context between one all the way up to context of block size.

And we train on that not just for computational reasons, because we happen to have the sequence already or something like that. It's not just done for efficiency.

It is also done to make the transformer network be used to seeing contexts all the way from as little as one all the way to block size.

And we like the transformer to be used to seeing everything in between.

And that's going to be useful later during inference because while we're sampling, we start the sampling generation with as little as one character of context.

And the transformer knows how to predict the next character with all the way just context of one.

And so that it can predict everything up to block size.

And after block size, we have to start truncating, because the transformer will never receive more than block size inputs when it's predicting the next character.

So we've looked at the time dimension of the tensors that are gonna be feeding into transformer. There's one more dimension to care about, and that is the batch dimension.

And as we're sampling these chunks of text, we're going to be actually, every time we're gonna feed them into a transformer, we're going to have many batches of multiple chunks of text that are all like stacked up in a single tensor. And it just done for efficiency, just so that we can keep the GPUs busy, because they are very good at parallel processing of data. And we just want to process multiple chunks all at the same time but those chunks are processed completely independently they don't talk to each other and so on so.

So let me just basically just generalize this and introduce a batch dimension, here's a chunk of code, let me just run it and then i'm going to explain what it does.

In [ ]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('input:')
print(xb.shape)
print(xb)
print('target:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f"when input is {context.tolist()} the target: {target}")

So here, we're going to start sampling random locations in the data set to pull chunks from, I am setting the seed in the random number generation so that the numbers I see here are going to be the same numbers you see later, if you try to reproduce this.

Now the `batch size` here is how many independent sequences we are processing every forward-backward pass of the transformer.

The `block size`, as I explained, is the maximum context length to make those prediction.

So let's say `batch size` 4, `block size` 8, and then here's how we get batch for any arbitrary split. If the split is a training split, then we're going to look at train data, otherwise at valid data. That gives us the data array. And then when I generate random positions to grab a chunk out of, I actually generate batch size number of random offset. So because this is four, `ix` is going to be four numbers that are randomly generated between zero and len of data minus block size. So it's just random offsets into the training set. And then `x`, as I explained, are the first block size characters starting at `i`. The y are the offset by one of that, so just add plus one. And then we're going to get those chunks for every one of integers i in ix and use a torch.stack to take all the those one-dimensional tensors, as we saw here, and we're going to stack them up as rows. And so they all become a row in a four by eight tensor.

When I sample a batch xb and yb, the inputs to the transformer now are the input x is the four by eight tensor, four rows of eight columns. And then 